# МТС · Qwen3-0.6B · настоящее корректирующее LoRA-дообучение

**1024 примера. Продолжаем +500.** Ячейка4 обучает модель; ячейка6 отдельно проверяет383 звонка и по умолчанию отключена. Старая модель сохраняется как запасная. Нужен один ZIP: лёгкий (если прежние веса уже доступны) или полный.

## 1. GPU, Drive и общий бюджет
Сначала дождитесь завершения прежней проверки на этой VM. Нужна A100/L4/T4; точная скорость измерится после первых обновлений. Для продолжения укажите RESUME_RUN; бюджет не сбрасывается.

In [ ]:
import json,time,sys,subprocess,os
from pathlib import Path
from datetime import datetime,timezone
from google.colab import drive,files
drive.mount('/content/drive')
TOTAL_MINUTES = 300
TRAIN_PHASE_MINUTES = 180
TRAIN_WINDOW_MINUTES = 90
ABSOLUTE_DEADLINE_MSK = '2026-11-16T06:15:00+03:00'
assert 120<=TOTAL_MINUTES<=300 and 60<=TRAIN_PHASE_MINUTES<=TOTAL_MINUTES-60 and 10<=TRAIN_WINDOW_MINUTES<=120
RESUME_RUN = ''
START_NEW_RUN = False
locator=Path('/content/mts_temporal_v23_session.json')
if RESUME_RUN:RUN=Path(RESUME_RUN)
elif locator.exists() and not START_NEW_RUN:RUN=Path(json.loads(locator.read_text())['run'])
else:RUN=Path('/content/drive/MyDrive/MTS_Qwen06_Temporal_runs')/datetime.now(timezone.utc).strftime('temporal_%Y%m%dT%H%M%SZ')
RUN.mkdir(parents=True,exist_ok=True);locator.write_text(json.dumps({'run':str(RUN)}))
budget_file=RUN/'session_budget.json'
if budget_file.exists():
    budget=json.loads(budget_file.read_text())
    assert budget['total_minutes']==TOTAL_MINUTES and budget['absolute_deadline']==ABSOLUTE_DEADLINE_MSK, 'Для продолжения сохраните прежние настройки'
else:
    origin=time.time();end=min(origin+TOTAL_MINUTES*60,datetime.fromisoformat(ABSOLUTE_DEADLINE_MSK).timestamp())
    budget={'origin_epoch':origin,'total_minutes':TOTAL_MINUTES,'deadline_epoch':end,'absolute_deadline':ABSOLUTE_DEADLINE_MSK,
            'train_phase_deadline_epoch':min(origin+TRAIN_PHASE_MINUTES*60,end-60*60)}
    budget_file.write_text(json.dumps(budget,indent=2))
assert budget['deadline_epoch']-time.time()>300,'Общий бюджет этого запуска исчерпан'
if budget['train_phase_deadline_epoch']-time.time()<=600:print('Предел обучения близок или истёк; остаётся только итоговая проверка уже готового export.')
print('Папка результатов:',RUN)
print('Общий остаток минут:',round((budget['deadline_epoch']-time.time())/60,1))
subprocess.run(['nvidia-smi'],check=True)

## 2. Загрузить один ZIP и восстановить исходный +500
Лёгкий пакет использует прежний ZIP/Drive текущего Colab. Полный уже содержит нужный адаптер и tokenizer. Если лёгкий пакет не нашёл веса, поставьте FORCE_UPLOAD=True и загрузите полный ZIP.

In [ ]:
import zipfile,hashlib,re,importlib.util
upload_dir=Path('/content/mts_v23_uploads');upload_dir.mkdir(exist_ok=True)
FORCE_UPLOAD = False
choices=['mts_qwen06_temporal_v23_full.zip','mts_qwen06_temporal_v23_light.zip']
available=[upload_dir/n for n in choices if (upload_dir/n).is_file()]
if not available or FORCE_UPLOAD:
    uploaded=files.upload()
    for name,data in uploaded.items():
        canonical=re.sub(r' (d+)(?=.zip$)','',Path(name).name)
        assert canonical in choices,'Нужен один из ZIP v23'
        (upload_dir/canonical).write_bytes(data)
    del uploaded
    available=[upload_dir/n for n in choices if (upload_dir/n).is_file()]
assert available,'Загрузите пакет v23'
payload=available[0]
expected_payloads={'mts_qwen06_temporal_v23_full.zip': 'b24bbf27ae97887c60297fff6a45737cb5ccb6892fa383f6bd5e9f2823bbede5', 'mts_qwen06_temporal_v23_light.zip': '7e81b2a4c266ca60aaedb9abd434bd9c38ae3605eb0c33777c988cb4a0d6d858'}
assert hashlib.sha256(payload.read_bytes()).hexdigest()==expected_payloads[payload.name], 'Неверная версия ZIP v23'
ROOT=Path('/content/mts_qwen06_temporal_v23');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(payload) as z:
    assert len(z.namelist())==len(set(z.namelist())) and sum(i.file_size for i in z.infolist())<200*1024**2
    for item in z.infolist():
        assert (ROOT/item.filename).resolve().is_relative_to(ROOT.resolve()) and chr(92) not in item.filename and ':' not in item.filename
    z.extractall(ROOT)
spec=importlib.util.spec_from_file_location('v23_restore_parent',ROOT/'restore_parent.py');restore=importlib.util.module_from_spec(spec);spec.loader.exec_module(restore)
print(restore.restore(ROOT))
manifest=json.loads((ROOT/'package_manifest.json').read_text())
for relative,want in manifest['sha256'].items():
    p=(ROOT/relative).resolve()
    assert p.is_relative_to(ROOT.resolve()) and hashlib.sha256(p.read_bytes()).hexdigest()==want,relative
print('Пакет и исходный +500 проверены:',len(manifest['sha256']),'файлов')

## 3. Установить библиотеки и проверить данные
CUDA PyTorch не переустанавливается. Проверки пока без обучения; настоящий GPU loss-test и длинный backward будут в ячейке4.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements.txt')],check=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT);env['TOKENIZERS_PARALLELISM']='false'
subprocess.run([sys.executable,str(ROOT/'verify_data.py'),'--root',str(ROOT)],cwd=ROOT,env=env,check=True)
subprocess.run([sys.executable,'-m','pytest',str(ROOT/'tests'),'-q'],cwd=ROOT,env=env,check=True)
print('Данные и CPU-проверки прошли. Следующая ячейка действительно дообучает модель.')

## 4. Дообучение + проверка до/после + выбор версии
Один проход1024 примеров, максимум256 обновлений. Чекпоинты сохраняются на Drive. Сначала получаем исходные метрики; затем идут строки update1/256... При повторном запуске выбирается последний checkpoint, а завершённый запуск не переобучается заново.

In [ ]:
RESUME_CHECKPOINT = ''
finished=(RUN/'experiment_report.json').is_file() and (RUN/'export/model_manifest.json').is_file()
if not finished:
    cmd=[sys.executable,'-u',str(ROOT/'train_temporal.py'),'--root',str(ROOT),'--run',str(RUN),'--train-minutes',str(TRAIN_WINDOW_MINUTES)]
    if RESUME_CHECKPOINT:cmd+=['--resume',RESUME_CHECKPOINT]
    elif (RUN/'last_pointer.json').exists():
        last=json.loads((RUN/'last_pointer.json').read_text());cmd+=['--resume',str(RUN/last['relative_path'])]
    env=os.environ.copy();env['PYTHONPATH']=str(ROOT);env['TOKENIZERS_PARALLELISM']='false'
    try:
        result=subprocess.run(cmd,cwd=ROOT,env=env,timeout=max(1,budget['train_phase_deadline_epoch']-time.time()+120))
        print('Код завершения:',result.returncode)
    except subprocess.TimeoutExpired:
        (RUN/'supervisor_stop.json').write_text(json.dumps({'reason':'Training phase wall-clock limit; worker terminated'}))
        print('Время этапа закончилось. Чекпоинты сохранены на Drive; при этом завершённое улучшение не заявляется.')
    finally:
        subprocess.run([sys.executable,str(ROOT/'pack_run.py'),str(RUN)],cwd=ROOT,check=True)
        if (RUN/'export/model_manifest.json').exists():subprocess.run([sys.executable,str(ROOT/'pack_run.py'),str(RUN),'--model'],cwd=ROOT,check=True)
else:print('Этот запуск уже завершён. Веса повторно не обучаются.')
result_zip=RUN.parent/(RUN.name+'_results.zip')
if result_zip.exists():files.download(str(result_zip))
print('Чекпоинты и выбранная модель:',RUN)

## 5. Прочитать итог и скачать небольшой отчёт
Это не обучение. Показываются метрики на24 реальных проверочных звонках; синтетический стресс хранится отдельно. selected_updates=0 означает, что выбран исходный +500. Для разработчика можно отдельно скачать model.zip.

In [ ]:
import json,sys,subprocess
from pathlib import Path
from google.colab import files,drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
if 'RUN' not in globals():
    RECOVER_RUN = ''
    locator=Path('/content/mts_temporal_v23_session.json')
    if RECOVER_RUN:RUN=Path(RECOVER_RUN)
    elif locator.exists():RUN=Path(json.loads(locator.read_text())['run'])
    else:raise RuntimeError('Укажите RECOVER_RUN: путь папки temporal_... на Drive')
report_path=RUN/'experiment_report.json'
if report_path.exists():
    r=json.loads(report_path.read_text())
    print('Новые обновления:',r['new_updates'],'Выбрана версия:',r['selected_updates'])
    print('Область: 24 реальных внутренних проверочных звонка; не официальный val383')
    print('ДО:',json.dumps(r['baseline_best']['metrics'],ensure_ascii=False))
    print('ПОСЛЕ:',json.dumps(r['selected']['metrics'],ensure_ascii=False))
    print('Откат:',r['rollback'],'Reload:',r['reload'])
elif (RUN/'failure.json').exists():print((RUN/'failure.json').read_text())
else:print('Итоговый отчёт ещё не готов. Не запускайте второе обучение параллельно.')
ROOT=Path('/content/mts_qwen06_temporal_v23')
if ROOT.exists():subprocess.run([sys.executable,str(ROOT/'pack_run.py'),str(RUN)],cwd=ROOT,check=True)
archive=RUN.parent/(RUN.name+'_results.zip')
if archive.exists():files.download(str(archive))
DOWNLOAD_MODEL = False
model_zip=RUN.parent/(RUN.name+'_model.zip')
if DOWNLOAD_MODEL and model_zip.exists():files.download(str(model_zip))
print('Модель для разработчика на Drive:',model_zip)

## 6. Отдельно: три ресурсных режима и официальный val383
По умолчанию отключено. Включайте после просмотра результата. Бюджет общий с обучением, он не начинается заново. Нужен полный набор383 для проверки порога; частичный отчёт не считается прохождением.

In [ ]:
RUN_FINAL_VALIDATION = False
if RUN_FINAL_VALIDATION:
    assert (RUN/'export/model_manifest.json').is_file(),'Сначала завершите обучение и reload выбранного export'
    budget=json.loads((RUN/'session_budget.json').read_text())
    assert budget['deadline_epoch']-time.time()>300,'Общий бюджет уже исчерпан'
    env=os.environ.copy();env['PYTHONPATH']=str(ROOT);env['TOKENIZERS_PARALLELISM']='false'
    try:
        result=subprocess.run([sys.executable,'-u',str(ROOT/'validate_export.py'),'--root',str(ROOT),'--run',str(RUN)],cwd=ROOT,env=env,
            timeout=max(1,budget['deadline_epoch']-time.time()+120))
        print('Код завершения официальной проверки:',result.returncode)
    except subprocess.TimeoutExpired:print('Достигнут общий предел; сохранённые результаты остаются на Drive')
    finally:
        subprocess.run([sys.executable,str(ROOT/'pack_run.py'),str(RUN)],cwd=ROOT,check=True)
        files.download(str(RUN.parent/(RUN.name+'_results.zip')))
else:print('Полная оценка пока выключена. Дообучение выполняется только в ячейке4.')